In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.table('workspace.bronze.erp_cust_az12')

#Transformations

In [0]:
display(df)

In [0]:
#trimming


for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
#check if dob is greater than current date

(df.filter(col("BDATE") > current_date())).count()

In [0]:
df = df.withColumn('BDATE', when(col("BDATE") > current_date(), None).otherwise(col("BDATE")))

In [0]:
#cleaning c_id because it will be merged with crm_cust in gold layer

df = df.withColumn('CID', when(col("CID").startswith('NAS'), substring(col('cid'), 4, length(col("CID"))))
                   .otherwise(col("CID")))

In [0]:
#Standardization of gender column

df = df.withColumn('GEN', when(upper(col("GEN")).isin('M', 'MALE'), 'Male')
                   .when(upper(col("GEN")).isin('F','FEMALE'), 'Female')
                   .otherwise("n/a"))

In [0]:
#reanming column

rename_col = {
    'CID': 'CUST_ID',
    'BDATE': 'DOB',
    'GEN': 'GENDER'
}

for old, new in rename_col.items():
    df = df.withColumnRenamed(old, new)


#Writing into Silver Table

In [0]:
df.write.mode('overwrite').saveAsTable('workspace.silver.erp_customers')